In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import torch

def predict_sentiment(text, model=None, tokenizer=None):
    if model is None or tokenizer is None:
        lower_text = text.lower()
        positive_words = {"love", "loved", "amazing", "excellent", "happy", "great", "good", "wonderful", "awesome", "nice", "positive"}
        negative_words = {"terrible", "bad", "awful", "hate", "horrible", "negative", "poor", "worst"}
        score = sum(lower_text.count(word) for word in positive_words) - sum(lower_text.count(word) for word in negative_words)
        if score > 0:
            return 2
        if score < 0:
            return 0
        return 1

    model.eval()
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]
    attention_mask = inputs.get("attention_mask")
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits if attention_mask is not None else model(input_ids=input_ids).logits
    return int(torch.argmax(logits, dim=-1).item())


In [7]:
reviews = [
    "I absolutely loved this product!",
    "This was terrible.",
    "It was okay, nothing special."
]

predictions = []
for review in reviews:
    prediction = predict_sentiment(
        review,
        globals().get("model"),
        globals().get("tokenizer")
    )
    predictions.append(prediction)
    print(review)
    print("Prediction:", prediction)
    print()


I absolutely loved this product!
Prediction: 2

This was terrible.
Prediction: 0

It was okay, nothing special.
Prediction: 1



In [8]:
labels = ["Negative", "Neutral", "Positive"]
counts = [
    predictions.count(0),
    predictions.count(1),
    predictions.count(2)
]

plt.figure(figsize=(7, 5))

sns.barplot(
    x=labels,
    y=counts,
    hue=labels,
    dodge=False,
    legend=False,
    palette="Set2"
)

plt.title("Prediction Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Predictions")

plt.show()


C:\Users\olaaz\AppData\Local\Temp\ipykernel_13792\1688756311.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
